# Ensemble Methods

Bagging, AdaBoost, Gradient Boosting, Voting. Binary classification on California housing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (BaggingClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, VotingClassifier,
                              RandomForestClassifier)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

data_df = pd.read_csv("housing.csv")
data_df['total_bedrooms'] = data_df['total_bedrooms'].fillna(data_df['total_bedrooms'].median())
data_df = pd.get_dummies(data_df, columns=['ocean_proximity'], dtype=float)

threshold = data_df['median_house_value'].median()
y = (data_df['median_house_value'] > threshold).astype(int).to_numpy()
X = data_df.drop(columns=['median_house_value']).to_numpy()

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, shuffle=True, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

## Bagging

In [ ]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=8, random_state=42),
    n_estimators=100, random_state=42, n_jobs=-1
)
bag.fit(X_train, y_train)
print(f"Bagging test acc: {accuracy_score(y_test, bag.predict(X_test)):.4f}")

## AdaBoost

In [ ]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=200, learning_rate=1.0, random_state=42
)
ada.fit(X_train, y_train)
print(f"AdaBoost test acc: {accuracy_score(y_test, ada.predict(X_test)):.4f}")

stages = list(ada.staged_score(X_test, y_test))
plt.figure(figsize=(9, 5))
plt.plot(range(1, len(stages) + 1), stages)
plt.xlabel('boosting round')
plt.ylabel('test accuracy')
plt.grid(alpha=0.3)
plt.show()

## Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42
)
gb.fit(X_train, y_train)
print(f"GBM test acc: {accuracy_score(y_test, gb.predict(X_test)):.4f}")

gb_stages = [accuracy_score(y_test, (s > 0).astype(int))
             for s in gb.staged_decision_function(X_test)]

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(gb_stages) + 1), gb_stages)
plt.xlabel('boosting round')
plt.ylabel('test accuracy')
plt.grid(alpha=0.3)
plt.show()

## Learning rate sweep (GBM)

In [ ]:
lrs = [0.01, 0.05, 0.1, 0.3, 0.5]
for lr in lrs:
    m = GradientBoostingClassifier(n_estimators=200, learning_rate=lr,
                                   max_depth=3, random_state=42)
    m.fit(X_train, y_train)
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test, m.predict(X_test))
    print(f"lr={lr:<5} train={tr:.4f} test={te:.4f}")

## Voting (hard and soft)

In [ ]:
estimators = [
    ('lr', LogisticRegression(max_iter=1000, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('knn', KNeighborsClassifier(n_neighbors=15)),
]

vote_hard = VotingClassifier(estimators=estimators, voting='hard', n_jobs=-1)
vote_hard.fit(X_train, y_train)
print(f"Hard voting test acc: {accuracy_score(y_test, vote_hard.predict(X_test)):.4f}")

vote_soft = VotingClassifier(estimators=estimators, voting='soft', n_jobs=-1)
vote_soft.fit(X_train, y_train)
print(f"Soft voting test acc: {accuracy_score(y_test, vote_soft.predict(X_test)):.4f}")

## Compare all

In [ ]:
models = {
    'Bagging': bag,
    'AdaBoost': ada,
    'GradientBoost': gb,
    'Voting (hard)': vote_hard,
    'Voting (soft)': vote_soft,
}

names, accs = [], []
for name, m in models.items():
    a = accuracy_score(y_test, m.predict(X_test))
    names.append(name)
    accs.append(a)
    print(f"{name:<16} {a:.4f}")

plt.figure(figsize=(9, 5))
plt.bar(names, accs, color='steelblue')
plt.ylabel('test accuracy')
plt.ylim(min(accs) - 0.01, max(accs) + 0.01)
plt.grid(alpha=0.3, axis='y')
plt.show()